# Compressed RNNs for Edge-Deployable Battery SOH Estimation (NASA data set)

**How to run:** Runtime > Change runtime type > CPU (GPU not needed) > then **Runtime > Run all**.
Total time: about 15-30 min. For a 3-minute test run, set `QUICK = True` in the first code cell.

Outputs (saved in the `outputs/` folder and downloaded at the end as `outputs.zip`):
`results_raw.csv`, `results_summary.csv`, `fig1_soh_fade.png`, `fig2_tradeoff.png`, `fig3_prediction.png`.

In [ ]:
# ---- 1. Setup and settings ----
import os, io, glob, zipfile, time, copy, random, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
import torch, torch.nn as nn
warnings.filterwarnings("ignore")

QUICK     = False                       # True = fast test run (1 seed, fewer epochs)
SEEDS     = [0] if QUICK else [0, 1, 2]
EPOCHS    = 60 if QUICK else 150
FT_EPOCHS = 20 if QUICK else 40         # fine-tuning epochs after pruning
HIDDENS   = [16, 32, 64]
KINDS     = ["LSTM", "GRU", "CNN-GRU"]
PRUNE_SPARSITY = 0.5
L         = 32                          # points on the voltage grid
V_HI, V_LO = 3.9, 3.5                   # partial-discharge window (volts)
RATED_AH  = 2.0
BATTERIES = ["B0005", "B0006", "B0007", "B0018"]
os.makedirs("outputs", exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
print("torch", torch.__version__)

In [ ]:
# ---- 2. Download and extract the NASA battery data ----
URL = "https://phm-datasets.s3.amazonaws.com/NASA/5.+Battery+Data+Set.zip"

def find_mats():
    out = {}
    for p in glob.glob("nasa_data/**/*.mat", recursive=True):
        n = os.path.basename(p)[:-4]
        if n in BATTERIES:
            out[n] = p
    return out

if len(find_mats()) < 4:
    os.makedirs("nasa_data", exist_ok=True)
    if not glob.glob("nasa_data/**/*.zip", recursive=True):
        os.system(f'wget -q -O nasa_data/battery.zip "{URL}"')
    for _ in range(3):                                   # also unpack nested zips
        for z in glob.glob("nasa_data/**/*.zip", recursive=True):
            d = z[:-4]
            if not os.path.isdir(d):
                try:
                    zipfile.ZipFile(z).extractall(d)
                except Exception as e:
                    print("could not unzip", z, e)

MATS = find_mats()
print("Found:", MATS)
if len(MATS) < 4:
    print("\nAUTOMATIC DOWNLOAD FAILED. Manual fix:")
    print("1) Download the NASA 'Battery Data Set' zip from the NASA Prognostics Data Repository page.")
    print("2) Unzip it on your computer and upload B0005.mat, B0006.mat, B0007.mat, B0018.mat below.")
    from google.colab import files
    up = files.upload()
    os.makedirs("nasa_data/manual", exist_ok=True)
    for fn in up:
        os.replace(fn, f"nasa_data/manual/{fn}")
    MATS = find_mats()
assert len(MATS) == 4, f"Missing batteries, found only {list(MATS)}"

In [ ]:
# ---- 3. Parse the .mat files and build the partial-discharge dataset ----
def cycle_to_sample(d):
    V = np.asarray(d["Voltage_measured"], float)
    T = np.asarray(d["Temperature_measured"], float)
    t = np.asarray(d["Time"], float)
    if len(V) < 20 or V.max() < V_HI or V.min() > V_LO:
        return None
    Vm = np.minimum.accumulate(V)                        # make voltage non-increasing
    i0 = int(np.argmax(Vm <= V_HI)); i1 = int(np.argmax(Vm <= V_LO))
    if i1 - i0 < 5:
        return None
    sv, st, sT = Vm[i0:i1+1], t[i0:i1+1] - t[i0], T[i0:i1+1]
    grid = np.linspace(V_LO, V_HI, L)                    # ascending for np.interp
    time_curve = np.interp(grid, sv[::-1], st[::-1])[::-1]   # index 0 = 3.9 V
    temp_curve = np.interp(grid, sv[::-1], sT[::-1])[::-1]
    s = np.stack([time_curve, temp_curve], axis=1)
    return s if np.all(np.isfinite(s)) else None

def load_battery(name, path):
    cycles = loadmat(path, simplify_cells=True)[name]["cycle"]
    X, y = [], []
    for c in cycles:
        if c["type"] != "discharge":
            continue
        try:
            d = c["data"]
            cap = float(np.ravel(d["Capacity"])[0])
            s = cycle_to_sample(d)
        except Exception:
            continue
        if s is None or not np.isfinite(cap):
            continue
        X.append(s); y.append(cap / RATED_AH * 100.0)
    return np.array(X, np.float32), np.array(y, np.float32)

DATA = {n: load_battery(n, p) for n, p in MATS.items()}
summary = pd.DataFrame({n: {"usable_cycles": len(DATA[n][1]),
                            "SOH_first_%": DATA[n][1][0], "SOH_last_%": DATA[n][1][-1]}
                        for n in BATTERIES}).T
print(summary.round(2))
summary.to_csv("outputs/dataset_summary.csv")

In [ ]:
# ---- 4. Figure 1: SOH fade ----
plt.figure(figsize=(6, 4))
for n in BATTERIES:
    plt.plot(DATA[n][1], label=n)
plt.xlabel("Discharge cycle"); plt.ylabel("SOH (%)"); plt.grid(alpha=.3); plt.legend()
plt.title("SOH fade of NASA cells")
plt.tight_layout(); plt.savefig("outputs/fig1_soh_fade.png", dpi=200); plt.show()

In [ ]:
# ---- 5. Models, scaling, training, compression, metrics ----
class RNNReg(nn.Module):
    def __init__(self, kind, in_dim, hidden):
        super().__init__()
        self.kind = kind
        if kind == "LSTM":
            self.rnn = nn.LSTM(in_dim, hidden)
        elif kind == "GRU":
            self.rnn = nn.GRU(in_dim, hidden)
        elif kind == "CNN-GRU":
            self.conv = nn.Sequential(nn.Conv1d(in_dim, 16, 3, padding=1), nn.ReLU())
            self.rnn = nn.GRU(16, hidden)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):                                 # x: (B, L, C)
        if self.kind == "CNN-GRU":
            x = self.conv(x.transpose(1, 2)).transpose(1, 2)
        out, _ = self.rnn(x.transpose(0, 1))              # (L, B, H)
        return self.head(out[-1]).squeeze(-1)

class Scaler:
    def fit(self, X, y):
        flat = X.reshape(-1, X.shape[-1])
        self.xm, self.xs = flat.mean(0), flat.std(0) + 1e-6
        self.ym, self.ys = y.mean(), y.std() + 1e-6
        return self
    def x(self, X): return torch.tensor((X - self.xm) / self.xs, dtype=torch.float32)
    def y(self, y): return torch.tensor((y - self.ym) / self.ys, dtype=torch.float32)
    def inv(self, p): return p * self.ys + self.ym

def make_masks(model, sparsity):
    ws = {n: p for n, p in model.named_parameters() if p.dim() > 1}
    allw = torch.cat([p.detach().abs().flatten() for p in ws.values()])
    k = max(1, int(sparsity * allw.numel()))
    thr = torch.kthvalue(allw, k).values
    return {n: (p.detach().abs() > thr).float() for n, p in ws.items()}

def apply_masks(model, masks):
    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in masks:
                p.mul_(masks[n])

def train(model, X, y, epochs, lr=3e-3, bs=32, masks=None):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    n = len(X)
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            b = perm[i:i + bs]
            opt.zero_grad()
            nn.functional.mse_loss(model(X[b]), y[b]).backward()
            opt.step()
            if masks:
                apply_masks(model, masks)
        sched.step()
    return model

def quantize_int8(model):
    return torch.ao.quantization.quantize_dynamic(
        copy.deepcopy(model).cpu(), {nn.LSTM, nn.GRU, nn.Linear}, dtype=torch.qint8)

def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    nz = sum(int((p != 0).sum()) for p in model.parameters())
    return tot, nz

def size_kb(model):
    try:
        buf = io.BytesIO(); torch.save(model.state_dict(), buf); return buf.tell() / 1024
    except Exception as e:                      # fallback if a quantized module cannot be serialized
        print("size fallback:", e); buf = io.BytesIO(); torch.save(model, buf); return buf.tell() / 1024

@torch.inference_mode()
def latency_ms(model, x1, n=200):
    model.eval()
    for _ in range(20): model(x1)
    ts = []
    for _ in range(n):
        t0 = time.perf_counter(); model(x1); ts.append((time.perf_counter() - t0) * 1e3)
    return float(np.median(ts))

@torch.inference_mode()
def predict(model, X, sc):
    model.eval(); return sc.inv(model(X).numpy())

def metrics(yt, yp):
    e = yp - yt
    return dict(RMSE=float(np.sqrt(np.mean(e**2))), MAE=float(np.mean(np.abs(e))),
                MAPE=float(np.mean(np.abs(e) / yt) * 100))
print("definitions OK")

In [ ]:
# ---- 6. Main experiment: leave-one-battery-out x models x widths x seeds x variants ----
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
torch.set_num_threads(1)
rows, preds_store = [], {}

def record(meta, variant, model, tot, nz, Xte_t, yte, sc, x1):
    p = predict(model, Xte_t, sc)
    r = dict(meta, variant=variant, **metrics(yte, p), params_total=tot, params_nonzero=nz,
             size_kb=size_kb(model), latency_ms=latency_ms(model, x1))
    r["sparse_size_kb"] = nz * 4 / 1024 if variant == "Pruned-50%" else np.nan
    rows.append(r)
    return p

t_start = time.time()
for test_b in BATTERIES:
    train_b = [b for b in BATTERIES if b != test_b]
    Xtr = np.concatenate([DATA[b][0] for b in train_b]); ytr = np.concatenate([DATA[b][1] for b in train_b])
    Xte, yte = DATA[test_b]
    sc = Scaler().fit(Xtr, ytr)
    Xtr_t, ytr_t, Xte_t = sc.x(Xtr), sc.y(ytr), sc.x(Xte)
    x1 = Xte_t[:1]

    # classical baselines (accuracy only)
    for name, mdl in [("Ridge", Ridge(alpha=1.0)), ("RandomForest", RandomForestRegressor(200, random_state=0, n_jobs=-1))]:
        mdl.fit(Xtr_t.numpy().reshape(len(Xtr_t), -1), ytr_t.numpy())
        p = sc.inv(mdl.predict(Xte_t.numpy().reshape(len(Xte_t), -1)))
        rows.append(dict(test=test_b, model=name, hidden=np.nan, seed=0, variant="sklearn", **metrics(yte, p),
                         params_total=np.nan, params_nonzero=np.nan, size_kb=np.nan, latency_ms=np.nan, sparse_size_kb=np.nan))

    for kind in KINDS:
        for h in HIDDENS:
            for seed in SEEDS:
                meta = dict(test=test_b, model=kind, hidden=h, seed=seed)
                set_seed(seed)
                m = RNNReg(kind, 2, h)
                train(m, Xtr_t, ytr_t, EPOCHS)
                tot, nz = count_params(m)
                p = record(meta, "FP32", m, tot, nz, Xte_t, yte, sc, x1)
                if seed == 0:
                    preds_store[(test_b, kind, h)] = p
                record(meta, "INT8", quantize_int8(m), tot, tot, Xte_t, yte, sc, x1)
                pm = copy.deepcopy(m)
                masks = make_masks(pm, PRUNE_SPARSITY); apply_masks(pm, masks)
                train(pm, Xtr_t, ytr_t, FT_EPOCHS, lr=1e-3, masks=masks)
                tot2, nz2 = count_params(pm)
                record(meta, "Pruned-50%", pm, tot2, nz2, Xte_t, yte, sc, x1)
                record(meta, "Pruned-50%+INT8", quantize_int8(pm), tot2, nz2, Xte_t, yte, sc, x1)
    print(f"fold {test_b} done  ({(time.time()-t_start)/60:.1f} min)")

df = pd.DataFrame(rows)
df.to_csv("outputs/results_raw.csv", index=False)
print(df.shape)

In [ ]:
# ---- 7. Summary tables (these are the numbers for the paper) ----
g = df.groupby(["model", "hidden", "variant"], dropna=False)
summ = g.agg(RMSE=("RMSE", "mean"), RMSE_std=("RMSE", "std"), MAE=("MAE", "mean"), MAPE=("MAPE", "mean"),
             params=("params_total", "mean"), nonzero=("params_nonzero", "mean"),
             size_kb=("size_kb", "mean"), sparse_kb=("sparse_size_kb", "mean"),
             latency_ms=("latency_ms", "mean")).reset_index()
summ.to_csv("outputs/results_summary.csv", index=False)
pd.set_option("display.width", 200, "display.max_rows", 200)
print("=== TABLE 1: FP32 and baselines ===")
print(summ[summ.variant.isin(["FP32", "sklearn"])].round(3).to_string(index=False))
print("\n=== TABLE 2: compression effect ===")
print(summ[~summ.variant.isin(["sklearn"])].sort_values(["model", "hidden", "variant"]).round(3).to_string(index=False))
print("\n=== Per-battery RMSE of FP32 models (mean over seeds) ===")
print(df[df.variant == "FP32"].groupby(["model", "hidden", "test"]).RMSE.mean().unstack().round(3))

In [ ]:
# ---- 8. Figure 2: accuracy-efficiency trade-off ----
nn_sum = summ[summ.variant != "sklearn"]
colors = {"FP32": "tab:blue", "INT8": "tab:orange", "Pruned-50%": "tab:green", "Pruned-50%+INT8": "tab:red"}
marks = {"LSTM": "o", "GRU": "s", "CNN-GRU": "^"}
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for (kind, var), s in nn_sum.groupby(["model", "variant"]):
    size_col = s.sparse_kb.where(s.sparse_kb.notna(), s.size_kb) if var == "Pruned-50%" else s.size_kb
    ax[0].scatter(size_col, s.RMSE, c=colors[var], marker=marks[kind], s=45, alpha=.85)
    ax[1].scatter(s.latency_ms, s.RMSE, c=colors[var], marker=marks[kind], s=45, alpha=.85)
ax[0].set_xlabel("Model size (kB; sparse size for pruned)"); ax[1].set_xlabel("CPU latency (ms, 1 thread)")
for a in ax: a.set_ylabel("RMSE (% SOH)"); a.grid(alpha=.3)
from matplotlib.lines import Line2D
h1 = [Line2D([], [], color=c, marker="o", ls="", label=v) for v, c in colors.items()]
h2 = [Line2D([], [], color="gray", marker=m, ls="", label=k) for k, m in marks.items()]
ax[1].legend(handles=h1 + h2, fontsize=8)
plt.tight_layout(); plt.savefig("outputs/fig2_tradeoff.png", dpi=200); plt.show()

In [ ]:
# ---- 9. Figure 3: predicted vs true SOH for the best FP32 model ----
best = summ[summ.variant == "FP32"].sort_values("RMSE").iloc[0]
bk, bh = best["model"], int(best["hidden"])
print("Best FP32 model:", bk, bh, "RMSE", round(best["RMSE"], 3))
fig, axs = plt.subplots(1, 4, figsize=(16, 3.4), sharey=True)
for a, b in zip(axs, BATTERIES):
    a.plot(DATA[b][1], "k-", label="True"); a.plot(preds_store[(b, bk, bh)], "r--", label=f"{bk}-{bh}")
    a.set_title(b); a.set_xlabel("Cycle"); a.grid(alpha=.3)
axs[0].set_ylabel("SOH (%)"); axs[0].legend()
plt.tight_layout(); plt.savefig("outputs/fig3_prediction.png", dpi=200); plt.show()

In [ ]:
# ---- 10. Zip and download everything ----
import shutil
shutil.make_archive("outputs", "zip", "outputs")
try:
    from google.colab import files
    files.download("outputs.zip")
except Exception:
    print("Download outputs.zip from the file browser on the left.")